# Test Grok 4.20 Reasoning trên Azure

Notebook kiểm tra deployment `grok-4-20-reasoning` thông qua Azure OpenAI-compatible endpoint. Cấu hình được đọc từ file `.env` ở thư mục gốc của repo.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / ".env").exists():
            return path
    raise FileNotFoundError("Không tìm thấy file .env của repo")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ENV_PATH = REPO_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "").strip()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").strip().rstrip("/")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "").strip()

config = {
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_DEPLOYMENT": AZURE_OPENAI_DEPLOYMENT,
}
missing = [name for name, value in config.items() if not value]
if missing:
    raise RuntimeError(f"Thiếu biến môi trường: {', '.join(missing)}")

if not AZURE_OPENAI_ENDPOINT.endswith("/openai/v1"):
    raise ValueError("AZURE_OPENAI_ENDPOINT phải kết thúc bằng /openai/v1")

client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
)

print(f"Đã đọc cấu hình từ: {ENV_PATH}")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Deployment: {AZURE_OPENAI_DEPLOYMENT}")
print("API key: đã cấu hình")


## 1. Kiểm tra deployment Grok

Cell này xác nhận deployment trong `.env` xuất hiện trong danh sách model mà endpoint Azure cung cấp.

In [ ]:
available_models = sorted(model.id for model in client.models.list())
grok_models = [model_id for model_id in available_models if "grok" in model_id.lower()]

print(f"Endpoint trả về {len(available_models)} model entries.")
print("Các model Grok có thể nhìn thấy:")
for model_id in grok_models:
    marker = " <-- đang chọn" if model_id == AZURE_OPENAI_DEPLOYMENT else ""
    print(f"- {model_id}{marker}")

if AZURE_OPENAI_DEPLOYMENT not in available_models:
    raise RuntimeError(
        f"Không tìm thấy deployment '{AZURE_OPENAI_DEPLOYMENT}' trong danh sách model."
    )

print("\nDeployment Grok đã sẵn sàng.")


## 2. Test chat cơ bản

In [ ]:
response = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": "Bạn là trợ lý kỹ thuật. Trả lời chính xác và ngắn gọn bằng tiếng Việt.",
        },
        {
            "role": "user",
            "content": "Chỉ trả lời đúng câu: Grok trên Azure đang hoạt động.",
        },
    ],
    max_tokens=200,
)

answer = response.choices[0].message.content
usage = response.usage
reasoning_tokens = getattr(usage.completion_tokens_details, "reasoning_tokens", 0) or 0

print("Phản hồi:", answer)
print(f"Input tokens: {usage.prompt_tokens}")
print(f"Output tokens: {usage.completion_tokens}")
print(f"Reasoning tokens: {reasoning_tokens}")
print(f"Total tokens: {usage.total_tokens}")


## 3. Test khả năng suy luận

Bài toán có đáp án đúng là **17 phút**. Cell kiểm tra Grok có trả về kết quả đó hay không.

In [ ]:
reasoning_response = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": "Giải bài toán logic bằng tiếng Việt, trình bày ngắn gọn và kết luận rõ ràng.",
        },
        {
            "role": "user",
            "content": (
                "Có 4 người cần qua cầu vào ban đêm. Họ mất lần lượt 1, 2, 5 và 10 phút. "
                "Mỗi lần tối đa 2 người qua cầu, phải mang theo một đèn pin và khi đi đôi thì "
                "thời gian bằng người chậm hơn. Hãy tìm thời gian tối thiểu để tất cả qua cầu."
            ),
        },
    ],
    max_tokens=800,
)

reasoning_answer = reasoning_response.choices[0].message.content
reasoning_usage = reasoning_response.usage
reasoning_token_count = (
    getattr(reasoning_usage.completion_tokens_details, "reasoning_tokens", 0) or 0
)

print(reasoning_answer)
print("\n--- Token usage ---")
print(f"Input tokens: {reasoning_usage.prompt_tokens}")
print(f"Output tokens: {reasoning_usage.completion_tokens}")
print(f"Reasoning tokens: {reasoning_token_count}")
print(f"Total tokens: {reasoning_usage.total_tokens}")

if "17" not in reasoning_answer:
    print("Cảnh báo: câu trả lời không chứa đáp án kỳ vọng 17 phút.")
else:
    print("PASS: Grok trả về đáp án kỳ vọng 17 phút.")
